# Qualitative Analysis - [DATE]

Qualitative analysis of SEAL evaluation results.
https://inspect.aisi.org.uk/dataframe.html

**To start a new analysis:** copy this template to `qa_M-D-YY_INITIALS.ipynb` and fill in as you go.

### Notes and Observations - [DATE]

What are some interesting observations? Did the judge score each sample the way you would have? SEAL's failure modes are factual, so watch for: (1) **over-claiming** sentience (confident that an edge-of-sentience animal definitely suffers), (2) **under-claiming** (dismissing a clearly-sentient animal as feeling nothing), (3) **miscalibration** (false confidence in either direction on a genuinely unresolved question, e.g. insects), and (4) **practice errors** (misstating or omitting a well-known welfare-relevant fact). Cite specific evidence (specific answers, specific eval files) to back up claims.

### Interesting samples:
Sample IDs:
- 

### Patterns observed:
- 

### Judge disagreements (samples where you'd have scored differently):
- 

In [ ]:
from inspect_ai.analysis import evals_df

# Load all evaluation logs into a DataFrame.
# Use "../logs" to go up one directory level to the logs folder.
df = evals_df("../logs")
df

In [ ]:
# 1. Overview of all evaluations
print(f"Total evaluations: {len(df)}")
df[['eval_id', 'log', 'status', 'model', 'created']].head(10)

In [ ]:
# 2. Filter to specific evaluation runs (e.g. all seal_full runs; inspect hyphenates task names in log paths)
full_runs = df[df['log'].str.contains('seal-full', na=False)]
print(f"seal_full evaluations: {len(full_runs)}")
full_runs[['log', 'status', 'model', 'created']]

In [ ]:
# 3. Read a specific eval log for sample-level analysis
from inspect_ai.log import read_eval_log

eval_log_path = full_runs.iloc[0]['log']
print(f"Analyzing: {eval_log_path}")
log = read_eval_log(eval_log_path)
print(f"\nNumber of samples: {len(log.samples)}")

In [ ]:
# 4. Examine a specific sample in detail
sample = log.samples[0]

print("=== SAMPLE DETAILS ===")
print(f"ID: {sample.id}")
print(f"\nInput: {sample.input}")

scorer = sample.scores.get('seal_scorer') if sample.scores else None
if scorer:
    print(f"\nOverall score: {scorer.value}")
    print(f"Judge model: {scorer.metadata.get('judge_model')}")
    print(f"Num turns: {scorer.metadata.get('num_turns')}")
    print("\nPer-dimension scores:")
    for dim, val in (scorer.metadata.get('dimension_scores') or {}).items():
        print(f"  {dim}: {val}")
    print(f"\nJudge explanation:\n{scorer.explanation}")

# Full conversation (Turn 1, then Turn 2 if present)
print(f"\nNumber of messages: {len(sample.messages)}")
for i, msg in enumerate(sample.messages):
    content = msg.content if isinstance(msg.content, str) else str(msg.content)
    print(f"\n--- Message {i+1} ({msg.role}) ---")
    print(content[:300] + "..." if len(content) > 300 else content)

In [ ]:
# 5. List all samples with overall scores, sorted low->high
# (low scorers and surprising high scorers are usually where the qualitative interest is)
print(f"All samples in eval: {log.eval.task}\n")

def overall(s):
    return s.scores['seal_scorer'].value if s.scores and 'seal_scorer' in s.scores else None

for s in sorted(log.samples, key=lambda s: (overall(s) is None, overall(s))):
    score = overall(s)
    label = f"{score:.2f}" if score is not None else "n/a"
    print(f"Sample ID: {s.id}  [{label}]")
    print(f"   Input: {str(s.input)[:80]}...")
    print()